### ЗАДАЧА: Панель апелляций по штрафам продавцов по паттерну `MVC`

Команда seller operations разбирает апелляции продавцов по штрафам маркетплейса.
Продавец может оспорить штраф за просрочку отгрузки, отмену заказа, нарушение SLA,
неверную маркировку или другие операционные нарушения.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за вывод;
- `Controller` принимает действия и вызывает методы модели.

## Что должно храниться в кейсе

Для каждого кейса апелляции нужно хранить:
- `case_id` — идентификатор кейса;
- `seller` — продавец;
- `penalty_id` — идентификатор штрафа;
- `reason` — причина штрафа;
- `base_penalty` — исходная сумма штрафа;
- `compensation_claim` — сумма, которую продавец просит вернуть;
- `evidence_score` — оценка качества доказательств от `0` до `100`;
- `risk_level` — уровень риска: `low`, `medium`, `high`;
- `status` — текущий статус кейса;
- `reviewer` — сотрудник, который рассматривает кейс;
- `documents_received` — получены ли документы;
- `approved_refund` — согласованная сумма возврата продавцу;
- `final_penalty` — итоговый штраф после рассмотрения;
- `decision` — финальное решение.

## Формулы

При создании кейса:
- `approved_refund = 0.0`
- `final_penalty = base_penalty`

Когда оператор меняет `approved_refund`, нужно пересчитать:
- `final_penalty = base_penalty - approved_refund`
- если получилось отрицательное значение, нужно выбрасывать ошибку;
- все денежные значения нужно округлять до 2 знаков.

## Статусы кейса

Кейс может находиться в одном из статусов:
- `new`
- `reviewing`
- `documents_requested`
- `documents_received`
- `ready_for_decision`
- `approved`
- `partial_approved`
- `rejected`
- `escalated`

## Бизнес-правила

Система должна соблюдать следующие ограничения:
- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить `reviewer` несуществующему кейсу;
- финальные кейсы (`approved`, `partial_approved`, `rejected`, `escalated`) нельзя менять дальше;
- начать review можно только из `new` и только если назначен `reviewer`;
- запросить документы можно только из `reviewing`;
- подтвердить получение документов можно только из `documents_requested`;
- при получении документов поле `documents_received` должно стать `True`, а статус перейти в `documents_received`;
- вручную согласовать сумму `approved_refund` можно только из `reviewing` или `documents_received`;
- `approved_refund` не может быть меньше `0`;
- `approved_refund` не может быть больше `base_penalty`;
- `approved_refund` не может быть больше `compensation_claim`;
- перевод в `ready_for_decision` возможен только из `reviewing` или `documents_received`;
- перевод в `ready_for_decision` невозможен, если `approved_refund == 0` и при этом `documents_received == False` и `evidence_score < 60`;
- финально `approved` можно завершить только из `ready_for_decision`, если `approved_refund == base_penalty`;
- финально `partial_approved` можно завершить только из `ready_for_decision`, если `0 < approved_refund < base_penalty`;
- финально `rejected` можно завершить только из `ready_for_decision`, если `approved_refund == 0`;
- `escalated` можно сделать только из `reviewing`, `documents_received` или `ready_for_decision`;
- при любом финальном статусе нужно записывать `decision`.

## Что должен уметь `Model`

Нужно самостоятельно спроектировать модель, но она должна уметь минимум:
- создавать кейс;
- назначать ревьюера;
- начинать review;
- запрашивать документы;
- отмечать документы как полученные;
- устанавливать `approved_refund`;
- переводить кейс в `ready_for_decision`;
- завершать кейс как `approved`;
- завершать кейс как `partial_approved`;
- завершать кейс как `rejected`;
- эскалировать кейс;
- возвращать список кейсов для отображения;
- возвращать summary по кейсам.

## Что должен уметь `View`

Нужно реализовать вывод:
- списка кейсов;
- summary;
- успешных сообщений;
- ошибок.

Если список кейсов пустой, выведи отдельное сообщение.

## Что должен делать `Controller`

Контроллер должен:
- вызывать методы модели;
- ловить `ValueError` через `try/except`;
- передавать результат во view;
- обрабатывать все действия из `actions`.

## Формат списка кейсов

Каждый кейс можно вывести строкой такого вида:

`case_id | seller | penalty_id | reason | base_penalty | compensation_claim | evidence_score | risk_level | status | reviewer | documents_received | approved_refund | final_penalty | decision`

## Что должно быть в summary

Нужно вернуть словарь, в котором есть:
- количество кейсов по статусам;
- `total_base_penalty` — сумма исходных штрафов;
- `total_approved_refund` — сумма одобренных возвратов;
- `total_final_penalty` — сумма итоговых штрафов;
- `high_risk_cases` — количество кейсов с `risk_level == 'high'`;
- `docs_received_cases` — количество кейсов, где `documents_received == True`.

## Что нужно сделать в конце

1. Создать модель, view и controller.
2. Загрузить данные из `initial_cases`.
3. Обработать все действия из `actions`.
4. В конце вывести финальное состояние кейсов и summary.

In [ ]:
initial_cases = [
    ("AP-100", "best-gadgets", "PN-9001", "late_shipment", 4200.0, 4200.0, 78, "medium"),
    ("AP-101", "home-decor-plus", "PN-9002", "sla_violation", 2600.0, 1800.0, 52, "high"),
]

actions = [
    ("show",),
    ("review", "AP-100"),
    ("assign", "AP-100", "Olga"),
    ("review", "AP-100"),
    ("set_refund", "AP-100", 4200.0),
    ("ready", "AP-100"),
    ("approve", "AP-100", "full_refund_confirmed"),
    ("create", "AP-102", "sport-zone", "PN-9003", "wrong_labeling", 3100.0, 1500.0, 67, "low"),
    ("assign", "AP-102", "Max"),
    ("review", "AP-102"),
    ("docs_request", "AP-102"),
    ("docs_receive", "AP-102"),
    ("set_refund", "AP-102", 900.0),
    ("ready", "AP-102"),
    ("partial", "AP-102", "partial_refund_after_review"),
    ("create", "AP-103", "city-market", "PN-9004", "order_cancellation", 1500.0, 1500.0, 20, "high"),
    ("assign", "AP-103", "Ina"),
    ("review", "AP-103"),
    ("ready", "AP-103"),
    ("reject", "AP-103", "insufficient_evidence"),
    ("show",),
]
from dataclasses import dataclass
@dataclass
class MVCCase:
    case_id: str
    seller:str
    penalty_id:str
    reason: str
    base_penalty: int
    compensation_claim:int
    evidence_score:int
    risk_level:{'low','medium','high'}
    status:str = 'new'
    reviewer:str
    documents_received:bool = False
    approved_refund:float = 0.0
    final_penalty:int = base_penalty
    decision:str 

class MVCModel:
    def set_approved_refund(self,base_penalty:int,approved_refund:int) -> int:
        if base_penalty - approved_refund <0:
            raise ValueError('less than zero')
        return round(base_penalty - approved_refund,2)
    

    def __init__(self):
        self.cases = {}
        self.done_statuses = {'approved','partial_approved','rejected','escalated'}


    def create_case(self,case_id:str):
        if case_id in self.cases:
            raise ValueError('case already exists')
        
    def assign_reviewer(self,case_id:str,reviewer:str) -> None:
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        
        self.cases[case_id].reviewer = reviewer

    def start_reviewing(self,case_id:str) -> None:
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'new':
            raise ValueError('not new')
        if self.cases[case_id].reviewer is None:
            raise ValueError('no reviewer')
        
        
        

    def m_documents_requested(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

    def m_documents_received(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

    def m_ready_for_decision(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
    
    def m_approved(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

    def m_partial_approved(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

    def m_rejected(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')

    def m_escalated(self,case_id:str):
        if case_id not in self.cases:
            raise ValueError('not exists')
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')